In [1]:
from functools import lru_cache
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats
import warnings

warnings.filterwarnings('ignore')

## 【自然出怪论】Part 1 变速

在自然出怪中，存在两个专门针对红眼的调控机制：

- 变速：一次选卡中红眼总数达到50只后，红眼不会在通常波出现
- 旗帜波：旗帜波红眼权重×6，导致实际红眼数约为通常波的3.5倍

这两个特殊机制（尤其是变速）使得对红眼数量的分析变得更加复杂。本文针对一些常见的场景给出了定量分析。
受能力与精力所限，难以涵盖所有相关内容。如有值得补充者欢迎指出。

自然出怪的具体机制此处不再赘述。见[PT站](https://pvz.tools/wiki/#%E5%88%97%E8%A1%A8)。

本文是一个Jupyter Notebook，图表全部由文中的代码生成。

文中的图是交互式的，可以放大、缩小，鼠标放在格点上会显示具体数值。多变量图表也可以选择可见的变量。

In [2]:
weights = {
    'PE': [0] * 2 + [0+1000j] + [1000+1000j] * 6 + [1500+1500j] * 3 + [2000+2000j] * 5 + [3000+3000j, 3500+3500j],
    'DE': [0] * 2 + [0+1000j] + [1000+1000j] * 6 + [1500+1500j] * 2 + [2000+2000j] * 4 + [3000+3000j, 3500+3500j],
    'NE': [0] * 2 + [0+1000j] + [1000+1000j] * 6 + [1500+1500j] * 2 + [2000+2000j] * 3 + [3000+3000j, 3500+3500j],
    'RE': [0] * 2 + [0+1000j] + [1000+1000j] * 4 + [1500+1500j] * 2 + [2000+2000j] * 4 + [3000+3000j, 3500+3500j]
}

def calc_weight_to_prob(weights, pick=8, base=2401+7401j):
    @lru_cache(maxsize=None)
    def dp(n, m):
        if m > n: return {}
        if m == 0: return {base: 1}
        result = dp(n - 1, m)
        for (w, c) in dp(n - 1, m - 1).items():
            w += weights[n - 1]
            if w not in result: result[w] = 0
            result[w] += c
        return result
    result = dp(len(weights), pick)
    total = sum(result.values())
    return {w: c / total for (w, c) in result.items()}

在研究选卡内红眼数量前，首先要了解出怪权重的分布。这一权重决定了每波的红眼数，也会影响总体分布（半场高密度 vs. 全红）。

虽然不同场地下权重的分布形状接近，但由于可能出怪种类不同，仍有较小差别。红眼出率排序为 PE≈RE<DE<NE。

含红眼关通常波权重在11901\~13401之间最为集中，然而权重极差却有13000之多。考虑到极端权重的出现概率（PE场地下，7400与20400权重的概率都不超过$10^{-4}$），仅分析极端情况于实践的指导性是有限的。后文中所有数据（除明确说明的）都是在所有可能权重中按概率加权的结果。

In [3]:
_data = []
for w in weights.values():
    w2p = calc_weight_to_prob(w)
    _data.append({w.real: p for (w, p) in w2p.items()})
_data = pd.DataFrame(_data, index=weights.keys()).T
px.bar(_data, barmode='group', labels=dict(index='权重', value='概率')).update_xaxes(tick0=7400, dtick=1000)

接下来是本文的核心——每波红眼数的分布。具体而言，得到的结果是三维数组 $\mathbf{P}$，$\mathbf{P}_{i, j, k}$ 代表到第 $i$ 波累计出 $j$ 个红眼，本波出 $k$ 个红眼的概率。
数据通过动态规划由二项分布的概率密度函数算出，忽略浮点误差的情况下绝对精确。

$\mathbf{P}$ 具有很强的通用性，可用于分析各种问题。本文主要聚焦以下三点：

1. 对红眼分布的图形化认识
2. 不同波次的变速概率
3. 收尾特化在概率视角上的可行性

若有缺漏，欢迎提出意见。

In [4]:
@lru_cache(maxsize=None)
def _binom_pmf(n, p, x): return scipy.stats.binom(n, p).pmf(x)
@lru_cache(maxsize=None)
def _binom_sf(n, p, x): return scipy.stats.binom(n, p).sf(x)

@lru_cache(maxsize=None)
def calc_giga_prob(weight):
    result = np.zeros((21, 133, 51))
    k_sum = np.zeros((21, 133))
    result[0][0][0] = k_sum[0][0] = 1
    for i in range(1, 21):
        huge = (i % 10 == 0)
        n = 41 if huge else 50
        p = (6000 / weight.imag) if huge else (1000 / weight.real)
        if not huge:
            for k in range(51):
                result[i][50][k] = k_sum[i-1][50-k] * _binom_sf(n, p, k - 1)
            for j in range(51, 133):
                result[i][j][0] = k_sum[i-1][j]
        for j in range(133 if huge else 50):
            for k in range(42 if huge else j+1):
                result[i][j][k] = k_sum[i-1][j-k] * _binom_pmf(n, p, k)
        k_sum[i] = result[i].sum(axis=1)
    return result[1:]

giga_prob = np.zeros((4, 20, 133, 51))
for (i, w) in enumerate(weights.values()):
    for (w, p) in calc_weight_to_prob(w).items():
        giga_prob[i] += calc_giga_prob(w) * p
total_giga_prob = giga_prob.sum(axis=3)
wave_giga_prob = giga_prob.sum(axis=2)

下面对数据的可信程度进行验证。将 $N = 449391$ 次试验中“出红眼波次总数”之和服从的分布近似为正态分布，构建 95% 置信区间，并取[PT站](https://pvz.tools/wiki/#%E5%88%97%E8%A1%A8)的数据进行对比。

In [5]:
_observed = [438301, 438326, 438441, 438320, 438310, 438297, 438239, 437725, 435309, 288111, 220008, 155411, 102375, 62629, 35997, 19409, 9853, 4655]
_p = wave_giga_prob[0, np.arange(20) % 10 != 9, 1:].sum(axis=1)
_mean = np.sum(449391 * _p)
_var = np.sum(449391 * _p * (1-_p))
_dist = scipy.stats.norm(_mean, _var ** 0.5)
print('95% 置信区间：', [_dist.ppf(0.025), _dist.ppf(0.975)])
print('测试值：', sum(_observed))

95% 置信区间： [4837089.731044252, 4840168.113487928]
测试值： 4839716


测试值落在 95% 置信区间内，说明无法得出计算值与实际情况有显著差异的结论。

下图是PE场地累计红眼数分布随波次变化的热力图。颜色轴为对数刻度（蓝色部分为 $\le 10^{-4}$），实际方差并非如此之大。

图中可以发现红眼数量变化的整体趋势：w9及以前基本是匀速增长（虽然也有小概率变速），w11开始逐渐向50红集中，大于50红的部分不再变化。w19大概率已经出满。

结果非常直观，也与普遍认知相符。

In [6]:
px.imshow(np.log10(np.maximum(total_giga_prob[0], 1e-4))[:, :101], labels=dict(x='累计红眼数', y='波数'), aspect='auto')

在不同权重下，图像的差异非常明显。如下是PE场地典型的低权重与高权重的分布图。可以看出9401权重时，w9出满的概率已经很高，w11出红眼的情况几乎不存在，总红眼数期望能达到80\-85；16401权重时增长缓慢得多，红眼总数也少了接近20。

In [7]:
px.imshow(np.log10(np.maximum(calc_giga_prob(9401+14401j)[:, :101].sum(axis=2), 1e-4)),
          labels=dict(x='累计红眼数（9401权重）', y='波数'), aspect='auto').show()
px.imshow(np.log10(np.maximum(calc_giga_prob(16401+21401j)[:, :101].sum(axis=2), 1e-4)),
          labels=dict(x='累计红眼数（16401权重）', y='波数'), aspect='auto').show()

取累计红眼数图像的第一行和最后一行单独绘制为折线图。各场地差异不大，以PE为例，每波红眼数 90% 区间为 1-7 红，总红眼数则为 58-77 红。

每波红眼数在自然出怪下非常不稳定，是刷新不稳定性最主要的来源。

In [8]:
_data = pd.DataFrame(wave_giga_prob[:, 0].T, columns=weights.keys())[:16]
px.line(_data, labels=dict(index='每波红眼数', value='概率')).show()
_data = pd.DataFrame(total_giga_prob[:, 19].T, columns=weights.keys())[50:101]
px.line(_data, labels=dict(index='总红眼数', value='概率')).show()

下图是“变速”（红眼累计达到50只）在各波发生的概率。w9变速的概率在 2.5%-4.8% 之间，超过95%情况下w16都已经变速，w18红眼还没出满的概率只有 0.57-1.16%。

In [9]:
_data = giga_prob[:, :19, 50:].sum(axis=(2, 3))
_data = pd.DataFrame(_data.T, index=range(1, 20), columns=weights.keys())
px.line(_data, labels=dict(index='波数', value='变速概率')).update_xaxes(dtick=1)

在键控炮阵和无炮中，都或多或少地存在着“收尾某行不出红眼”的特化。结合之前的红眼分布数据以及简单的数学计算，可以得出不同收尾特化对应的概率。“樱桃”指的是某波出的红眼在相邻三行。

在w19，不出红眼的概率就已经非常高，各种特化都达到了99%级别的可行性。w9的情况较为复杂，但PE的三行和五行场地的四行都是可行性比较高的。五行场地樱桃覆盖全部红眼的概率意外地不低。

In [10]:
def calc_dist_prob(prob_data, formula):
    n, coeff = formula
    x = np.arange(51.0)
    result = np.zeros(51)
    for (i, k) in enumerate(coeff):
        result += k * np.power((i + 1) / n, x)
    result[0] = 1
    return np.sum(result * prob_data)
_cdp = lambda s, w, f: calc_dist_prob(giga_prob[s][w-1].sum(axis=0), f)

_data = {}
for w in (9, 19):
    _data[('PE', w)] = []
    _data[('PE', w)].append(_cdp(0, w, (4, [])))
    _data[('PE', w)].append(_cdp(0, w, (4, [4])))
    _data[('PE', w)].append(_cdp(0, w, (4, [-8, 6])))
    _data[('PE', w)].append(_cdp(0, w, (4, [4, -6, 4])))
    _data[('PE', w)].append(1)
    for i in range(4):
        _data[('PE', w)].append(_cdp(0, w, (4, [0] * i + [1])))
    _data[('PE', w)].append(_cdp(0, w, (4, [0, 2])))
for i in range(1, 4):
    scene = list(weights.keys())[i]
    for w in (9, 19):
        _data[(scene, w)] = []
        _data[(scene, w)].append(_cdp(i, w, (5, [])))
        _data[(scene, w)].append(_cdp(i, w, (5, [5])))
        _data[(scene, w)].append(_cdp(i, w, (5, [-15, 10])))
        _data[(scene, w)].append(_cdp(i, w, (5, [15, -20, 10])))
        _data[(scene, w)].append(_cdp(i, w, (5, [-5, 10, -10, 5])))
        for j in range(4):
            _data[(scene, w)].append(_cdp(i, w, (5, [0] * j + [1])))
        _data[(scene, w)].append(_cdp(i, w, (5, [0, -2, 3])))
pd.DataFrame(_data, index=['不出', '一行', '两行', '三行', '四行', '固定一行', '固定两行', '固定三行', '固定四行', '樱桃']).T

不出        一行        两行        三行        四行      固定一行      固定两行  \
PE 9   0.031658  0.184399  0.527861  0.871797  1.000000  0.069844  0.165273   
   19  0.989665  0.994469  0.998260  0.999773  1.000000  0.990866  0.992699   
DE 9   0.033050  0.164680  0.456064  0.780609  0.960578  0.059376  0.114840   
   19  0.992266  0.995732  0.998458  0.999692  0.999975  0.992959  0.993925   
NE 9   0.035129  0.160855  0.446066  0.772473  0.958299  0.060275  0.113941   
   19  0.994554  0.997020  0.998928  0.999786  0.999982  0.995047  0.995731   
RE 9   0.031513  0.171204  0.470504  0.791881  0.963663  0.059451  0.117319   
   19  0.988973  0.993879  0.997781  0.999555  0.999964  0.989954  0.991325   

           固定三行      固定四行        樱桃  
PE 9   0.403929  1.000000  0.298887  
   19  0.995542  1.000000  0.995733  
DE 9   0.231897  0.478996  0.466012  
   19  0.995287  0.997225  0.998010  
NE 9   0.228769  0.474564  0.458425  
   19  0.996692  0.998054  0.998613  
RE 9   0.237255  0.485753  0.477127  
   19  0.993264  0.996030  0.997143

扩展上表，可定义无法处理的出怪事件（失败事件），自动计算收尾率

用`lambda rows:`表达式定义失败事件，`rows`表示出红行的集合，示例如下
| 定义 | 解释 |
| --- | --- |
| `{3, 5} <= rows` | 失败事件为，3、5路同时出红 |
| `not (len(rows) <= 2)` | 接受事件为，出红行数不多于2，对应上表“两行” |
| `not (rows <= {1,2})` | 接受事件为，出红行不超出固定两行，对应上表“固定两行”  |

In [ ]:
from math import comb

def calc_finish(scene, wave, fail):
    n = 4 if scene == 'PE' else 5
    assert not fail(set())
    states = [
        {r + 1 for r in range(n) if mask >> r & 1}
        for mask in range(1 << n)
    ]
    formula = [
        sum((-1)**(len(rows)-m) * comb(len(rows), m)
            for rows in states if len(rows) >= m and not fail(rows))
        for m in range(1, n + 1)
    ]
    return _cdp(list(weights).index(scene), wave, (n, formula))

fail = lambda rows: {3, 5} <= rows
calc_finish('NE', 19, fail)

In [43]:
[
    # 失败：NE w17 3路出红
    calc_finish('NE', 17, lambda rows: 3 in rows),
    # 失败：MGE(DE) w17 3路出红
    calc_finish('DE', 17, lambda rows: 3 in rows),
    # 接受：NE w19 至多3路出红或至多5路出红
    calc_finish('NE', 19, lambda rows: not (rows <= {3} or rows <= {5})),
    # 接受：MGE(DE) w19 至多3路出红或至多5路出红
    calc_finish('DE', 19, lambda rows: not (rows <= {3} or rows <= {5})),
]

[0.9900883589889475,
 0.9870365769913285,
 0.9955404016190227,
 0.9936526270405229]

## 有、无舞王的权重与红眼分布

在选中红眼的条件下按是否选中舞王拆分。DE、NE 的舞王条件概率分别为 $8/17$、$8/16$；有舞王时从剩余类型中选 7 类，无舞王时选 8 类。两个 `giga_prob` 数组沿用原来的四维格式。

In [ ]:
giga_dancer_scenes = ('DE', 'NE')
giga_dancer_weight_prob, giga_no_dancer_weight_prob = {}, {}
giga_dancer_giga_prob = np.zeros((2, 20, 133, 51))
giga_no_dancer_giga_prob = np.zeros_like(giga_dancer_giga_prob)
for i, scene in enumerate(giga_dancer_scenes):
    w = weights[scene].copy()
    w.remove(1000 + 1000j)
    giga_dancer_weight_prob[scene] = calc_weight_to_prob(w, 7, 3401+8401j)
    giga_no_dancer_weight_prob[scene] = calc_weight_to_prob(w)
    for w, p in giga_dancer_weight_prob[scene].items():
        giga_dancer_giga_prob[i] += calc_giga_prob(w) * p
    for w, p in giga_no_dancer_weight_prob[scene].items():
        giga_no_dancer_giga_prob[i] += calc_giga_prob(w) * p
giga_dancer_weights = sorted({w.real for d in giga_dancer_weight_prob.values() for w in d})
giga_dancer_total_giga_prob = giga_dancer_giga_prob.sum(axis=3)
giga_dancer_wave_giga_prob = giga_dancer_giga_prob.sum(axis=2)

_data = [pd.Series(d).groupby(lambda w: w.real).sum() for d in giga_dancer_weight_prob.values()]
_data = pd.DataFrame(_data, index=giga_dancer_scenes).T.fillna(0)
px.bar(_data, barmode='group', labels=dict(index='权重', value='概率')).update_xaxes(tick0=7400, dtick=1000)

## 普通波选行平滑的马尔可夫稳态

状态为 $H=((L_1,S_1),\ldots,(L_5,S_5))$。


普通僵尸可选五行，舞王只选第 2、3、4 行。相同普通波权重共用一次稳态计算。

In [ ]:
def calc_stationary_row_prob(p, seed, n=2048, burn=512, sample=512):
    rng = np.random.default_rng(seed)
    last, second = np.zeros((2, n, 5), dtype=np.int32)
    index = np.arange(n)

    def row_prob(q, allowed=1):
        a = q * np.clip(1.5*q*(last+1) + .25*q*(second+1) - 1, .01, 100) * allowed
        return a / a.sum(axis=1, keepdims=True)

    def step():
        dancer = rng.random(n) < p
        allowed = np.ones((n, 5), dtype=bool)
        allowed[dancer, 0] = allowed[dancer, 4] = False
        q = np.where(dancer, 1 / 3, 1 / 5)[:, None]
        row = (rng.random(n)[:, None] > np.cumsum(row_prob(q, allowed), axis=1)).sum(axis=1)
        row = np.minimum(row, 4)
        last[allowed] += 1
        second[allowed] += 1
        second[index, row] = last[index, row]
        last[index, row] = 0

    for _ in range(burn): step()
    result = np.zeros(5)
    for _ in range(sample):
        result += row_prob(1 / 5).sum(axis=0)
        step()
    result /= n * sample
    result[[0, 4]] = result[[0, 4]].mean()
    result[1:4] = result[1:4].mean()
    return result

_row_seeds = np.random.SeedSequence(20260714).spawn(len(giga_dancer_weights))
giga_dancer_row_prob_by_weight = {
    w: calc_stationary_row_prob(1000 / w, seed)
    for w, seed in zip(giga_dancer_weights, _row_seeds)
}
giga_dancer_row_prob = np.stack([
    sum(p * giga_dancer_row_prob_by_weight[w.real] for w, p in d.items())
    for d in giga_dancer_weight_prob.values()
])
pd.DataFrame(giga_dancer_row_prob, index=giga_dancer_scenes, columns=range(1, 6))

## 舞王修正后的红眼收尾表

无舞王组合使用五行均匀边际，有舞王组合使用上面的稳态边际；二者分别在相同权重内合并红眼数量分布，再按 $P(D\mid G)$ 混合。E、M 分别表示边路和中间行。

这里仍采用“稳态边际 + 红眼间独立”的近似；精确版需要传播完整有序类型流。

In [ ]:
_masks = np.arange(32)

def calc_mask_prob(p):
    result = np.zeros((51, 32))
    result[0, 0] = 1
    for x in range(1, 51):
        for row, q in enumerate(p):
            np.add.at(result[x], _masks | (1 << row), result[x - 1] * q)
    return result

_bits = np.array([bin(x).count('1') for x in _masks])
_subset = lambda x: (_masks | x) == x
_finish_names = (
    '不出', '一行', '两行', '三行', '四行',
    '固定一行(E)', '固定一行(M)', '固定两行(EE)', '固定两行(EM)', '固定两行(MM)',
    '固定三行(EEM)', '固定三行(EMM)', '固定三行(MMM)', '固定四行(EEMM)', '固定四行(EMMM)', '樱桃'
)
_finish_masks = (
    _masks == 0, *(_bits <= n for n in range(1, 5)),
    *(_subset(x) for x in (1, 2, 17, 3, 6, 19, 7, 14, 23, 15)),
    _subset(7) | _subset(14) | _subset(28)
)

def calc_finish_prob(count, row):
    mask = count @ calc_mask_prob(row)
    return np.array([mask[keep].sum() for keep in _finish_masks])

def calc_giga_finish(scene, wave):
    i = giga_dancer_scenes.index(scene)
    dancer = sum(
        p * calc_finish_prob(
            calc_giga_prob(w)[wave - 1].sum(axis=0),
            giga_dancer_row_prob_by_weight[w.real]
        )
        for w, p in giga_dancer_weight_prob[scene].items()
    )
    no_dancer = calc_finish_prob(
        giga_no_dancer_giga_prob[i, wave - 1].sum(axis=0),
        np.full(5, 1 / 5)
    )
    q = 8 / len(weights[scene])
    return q * dancer + (1 - q) * no_dancer

_data = {
    (scene, wave): calc_giga_finish(scene, wave)
    for scene in giga_dancer_scenes for wave in (9, 19)
}
giga_finish_table = pd.DataFrame(_data, index=_finish_names).T
assert np.all((0 <= giga_finish_table) & (giga_finish_table <= 1))
giga_finish_table